# IndicWhisper Evaluation on Kathbath-Tamil
### Accelerator = GPU T4

# Cell 1: Install dependencies

In [1]:

!pip install editdistance -q
!pip install joblib tqdm librosa soundfile indicnlp indic-nlp-library -q
!pip install git+https://github.com/anoopkunchukuttan/indic_nlp_library.git -q
!pip install git+https://github.com/huggingface/transformers -q
!pip install git+https://github.com/huggingface/datasets -q
!pip install evaluate accelerate -q
!sudo apt install ffmpeg -y -q
print("All dependencies installed!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 56.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.1/121.1 kB 7.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 9.3 MB/s eta 0:00:00ta 0:00:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 100.9 MB/s eta 0:00:0000:01:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.

# Cell 2: Clone vistaar repo 

In [2]:

import os

if not os.path.exists('/kaggle/working/vistaar'):
    !git clone https://github.com/AI4Bharat/vistaar.git /kaggle/working/vistaar
    print("Cloned!")
else:
    print("Already cloned, skipping.")

%cd /kaggle/working/vistaar

Cloning into '/kaggle/working/vistaar'...
remote: Enumerating objects: 166, done.
remote: Counting objects: 100% (21/21), done.
remote: Compressing objects: 100% (21/21), done.
remote: Total 166 (delta 10), reused 0 (delta 0), pack-reused 145 (from 1)
Receiving objects: 100% (166/166), 61.78 KiB | 1.93 MiB/s, done.
Resolving deltas: 100% (95/95), done.
Cloned!
/kaggle/working/vistaar


# Cell 3: Write fixed evaluation.py

In [3]:

fixed_eval = '''
import argparse
from transformers import pipeline
import editdistance
from joblib import Parallel, delayed
from tqdm import tqdm
import json
import soundfile as sf
from torch.utils.data import Dataset
from indicnlp.normalize.indic_normalize import IndicNormalizerFactory
import os
import string
import re
import time
import torch

from transformers import (
    AutoConfig,
    AutoFeatureExtractor,
    AutoModelForSpeechSeq2Seq,
    AutoProcessor,
    AutoTokenizer,
)

lang_to_code = {
    "hindi": "hi",
    "sanskrit": "sa",
    "bengali": "bn",
    "tamil": "ta",
    "telugu": "te",
    "gujarati": "gu",
    "kannada": "kn",
    "malayalam": "ml",
    "marathi": "mr",
    "odia": "or",
    "punjabi": "pa",
    "urdu": "ur",
}


def normalize_sentence(sentence, lang_code):
    factory = IndicNormalizerFactory()
    normalizer = factory.get_normalizer(lang_code)
    return normalizer.normalize(sentence)


def compute_cer(refs, hyps):
    """Character Error Rate using editdistance (no rapidfuzz dependency)"""
    total_chars = 0
    total_edits = 0
    for r, h in zip(refs, hyps):
        total_edits += editdistance.eval(r, h)
        total_chars += len(r)
    return total_edits / total_chars if total_chars > 0 else 0.0


def compute_wer(refs, hyps):
    """Word Error Rate using editdistance (no rapidfuzz dependency)"""
    total_words = 0
    total_edits = 0
    for r, h in zip(refs, hyps):
        r_words = r.split()
        h_words = h.split()
        total_edits += editdistance.eval(r_words, h_words)
        total_words += len(r_words)
    return total_edits / total_words if total_words > 0 else 0.0


class eval_dataset(Dataset):
    def __init__(self):
        self.audios = []
        self.sents = []

    def __len__(self):
        return len(self.audios)

    def __getitem__(self, i):
        return {
            "raw": self.audios[i]["array"],
            "sampling_rate": self.audios[i]["sampling_rate"],
            "reference": self.sents[i],
            "path": self.audios[i]["path"],
            "duration": self.audios[i]["duration"],
        }

    def fill_data(self, aud, sent):
        self.audios.append(aud)
        self.sents.append(sent)


def get_data(split):
    js_data = json.loads(split)
    aud = {}
    # Use audio_filepath directly from manifest (no hardcoded path replacement)
    aud["path"] = js_data["audio_filepath"]
    y, sr = sf.read(aud["path"])
    aud["duration"] = js_data["duration"]
    aud["array"] = y
    aud["sampling_rate"] = sr
    return (aud, js_data["text"])


def main(args):
    with open(args.manifest_path, "r") as f:
        data = f.read()
        splits = data.split("\\n")
        if splits[-1] == "":
            splits = splits[:-1]

    da = Parallel(n_jobs=4)(delayed(get_data)(split) for split in tqdm(splits))

    dataset = eval_dataset()
    for d in da:
        dataset.fill_data(d[0], d[1])

    whisper_asr = pipeline(
        "automatic-speech-recognition",
        model=args.model_path,
        device=args.device,
    )

    if args.lang_code == "or":
        whisper_asr.model.config.forced_decoder_ids = (
            whisper_asr.tokenizer.get_decoder_prompt_ids(
                language=None, task="transcribe"
            )
        )
    else:
        whisper_asr.model.config.forced_decoder_ids = (
            whisper_asr.tokenizer.get_decoder_prompt_ids(
                language=args.lang_code, task="transcribe"
            )
        )

    hypothesis = []
    ground_truth = []

    os.makedirs(dir_path + "/predictions", exist_ok=True)
    out_name = (
        args.model_path.rsplit("/", 1)[-1]
        + "_"
        + args.manifest_name
        + "_predictions.json"
    )
    open(dir_path + "/predictions/" + out_name, "w").close()

    st = time.time()

    for out in tqdm(whisper_asr(dataset, batch_size=args.batch_size), total=len(dataset)):
        hyp = out["text"]
        ref = out["reference"][0]
        hyp = hyp.translate(str.maketrans("", "", string.punctuation + "\u0964\u06d4\u2019-\u0965"))
        ref = ref.translate(str.maketrans("", "", string.punctuation + "\u0964\u06d4\u2019-\u0965"))
        if args.lang_code[:2] != "ur":
            hyp = normalize_sentence(hyp, args.lang_code[:2])
            ref = normalize_sentence(ref, args.lang_code[:2])
        hyp = re.sub(" +", " ", hyp).strip()
        ref = re.sub(" +", " ", ref).strip()
        if ref == "":
            ref = "<empty>"
        hypothesis.append(hyp)
        ground_truth.append(ref)
        res = {
            "audio_filepath": out["path"][0],
            "duration": out["duration"][0],
            "text": ref,
            "pred_text": hyp,
        }
        with open(dir_path + "/predictions/" + out_name, "a") as f:
            json.dump(res, f, ensure_ascii=False)
            f.write("\\n")

    et = time.time()

    result = {}
    result["model"] = args.model_path
    result["dataset"] = args.manifest_name
    result["language"] = args.lang_code
    result["cer"] = compute_cer(ground_truth, hypothesis)
    result["wer"] = compute_wer(ground_truth, hypothesis)
    result["time_minutes"] = round((et - st) / 60, 2)
    result["batch_size"] = args.batch_size

    print("\\n===== RESULTS =====")
    for k, v in result.items():
        print(f"  {k}: {v}")

    with open(dir_path + "/results.csv", "a") as results_fp:
        print(",".join([str(v) for v in result.values()]), file=results_fp)


if __name__ == "__main__":
    dir_path = os.path.dirname(os.path.realpath(__file__))
    parser = argparse.ArgumentParser()
    parser.add_argument("--model_path", type=str, required=True)
    parser.add_argument("--manifest_path", type=str, required=True)
    parser.add_argument("--manifest_name", type=str, required=True)
    parser.add_argument("--device", type=int, default=-1)
    parser.add_argument("--batch_size", type=int, default=32)
    parser.add_argument("--language", type=str, required=True)
    args = parser.parse_args()

    if len(args.language) == 2:
        args.lang_code = args.language.lower()
    else:
        args.lang_code = lang_to_code[args.language.lower()]

    main(args)
'''

with open('/kaggle/working/vistaar/evaluation.py', 'w') as f:
    f.write(fixed_eval)

print("evaluation.py written successfully!")
!grep -n 'compute_cer\|compute_wer\|audio_filepath\|editdistance' /kaggle/working/vistaar/evaluation.py

evaluation.py written successfully!
4:import editdistance
47:def compute_cer(refs, hyps):
48:    """Character Error Rate using editdistance (no rapidfuzz dependency)"""
52:        total_edits += editdistance.eval(r, h)
57:def compute_wer(refs, hyps):
58:    """Word Error Rate using editdistance (no rapidfuzz dependency)"""
64:        total_edits += editdistance.eval(r_words, h_words)
94:    # Use audio_filepath directly from manifest (no hardcoded path replacement)
95:    aud["path"] = js_data["audio_filepath"]
164:            "audio_filepath": out["path"][0],
179:    result["cer"] = compute_cer(ground_truth, hypothesis)
180:    result["wer"] = compute_wer(ground_truth, hypothesis)


# Cell 4: Create FULL manifest

In [8]:

import json, os

BASE = "/kaggle/input/datasets/monuj007/kathbath-benchmarks"
IN_MANIFEST = f"{BASE}/kathbath/tamil/manifest.json"
OUT_MANIFEST = "/kaggle/working/manifest_tamil_full.json"

count = 0
with open(IN_MANIFEST) as f, open(OUT_MANIFEST, "w") as out:
    for line in f:
        line = line.strip()
        if not line:
            continue
        entry = json.loads(line)
        filename = os.path.basename(entry["audio_filepath"])
        entry["audio_filepath"] = f"{BASE}/kathbath/tamil/wavs/wavs/{filename}"
        out.write(json.dumps(entry, ensure_ascii=False) + "\n")
        count += 1

print(f"Created full manifest with {count} samples: {OUT_MANIFEST}")

Created full manifest with 1642 samples: /kaggle/working/manifest_tamil_full.json


# Cell 6: Run evaluation (OOM-safe) 

In [10]:

import os, torch

# Free GPU memory before starting
torch.cuda.empty_cache()
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Free memory: {torch.cuda.mem_get_info()[0]/1024**3:.2f} GB")

%cd /kaggle/working/vistaar

# Set env var to reduce memory fragmentation
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

!PYTORCH_ALLOC_CONF=expandable_segments:True python evaluation.py \
    --model_path="tamil_models/whisper-medium-ta_alldata_multigpu" \
    --manifest_path="/kaggle/working/manifest_tamil_full.json" \
    --manifest_name="kathbath" \
    --device=0 \
    --batch_size=4 \
    --language="ta"

GPU: Tesla T4
Free memory: 14.46 GB
/kaggle/working/vistaar
  0%|                                                  | 0/1642 [00:00<?, ?it/s][transformers] Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
[transformers] Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> 

# Cell 7: View results 

In [11]:
# ── ──────────────────────────────────────────────────────
print("=== Results CSV ===")
!cat /kaggle/working/vistaar/results.csv

print("\n=== Predictions ===")
!cat /kaggle/working/vistaar/predictions/whisper-medium-ta_alldata_multigpu_kathbath_predictions.json

=== Results CSV ===
tamil_models/whisper-medium-ta_alldata_multigpu,kathbath,ta,0.05707762557077625,0.3333333333333333,0.41,10
tamil_models/whisper-medium-ta_alldata_multigpu,kathbath,ta,0.04476341695776437,0.2498150571900074,60.25,4

=== Predictions ===
{"audio_filepath": "/kaggle/input/datasets/monuj007/kathbath-benchmarks/kathbath/tamil/wavs/wavs/844424930700429-685-f.wav", "duration": 10.031, "text": "வலிந்து காணாமல் ஆக்கப்பட்டவர்களது போராட்டத்தை சிதைக்க முற்படும் அரச புலனாய்வுக்கு துணை செல்பவர்களிடம் எச்சரிக்கையா இருக்குமாறு", "pred_text": "வலிந்து காணாமல் ஆக்கப்பட்டவர்களது போராட்டத்தை சிதைக்க முற்படும் அரச புலனாய்வுக்கு துணை செல்பவர்களிடம் எச்சரிக்கையாயிருக்குமாறு"}
{"audio_filepath": "/kaggle/input/datasets/monuj007/kathbath-benchmarks/kathbath/tamil/wavs/wavs/844424930447974-877-f.wav", "duration": 9.961375, "text": "மன்னிப்பென்ற வார்த்தையே உபயோகிக்கல புண்படும்படி பேசியதற்கு வருத்தங்கள்னு தான் சொன்னாரு தமிழன் தமிழை காப்பதுபோல் எம்தமிழும் தமிழனை காக்கும்", "pred_text": "மன்னிப்ப